# Fetch Monthly Crude Oil Production for Non-OPEC Countries

## Goal

This notebook downloads **monthly crude oil production by country** from the U.S. Energy Information Administration (EIA) International API, removes current OPEC member countries, and creates analysis-ready datasets.

### Outputs

1. **Long format** — one row per month × country.
2. **Wide format** — one row per month, one production column per non-OPEC country.
3. **Monthly non-OPEC total** — sum of the country-level series.
4. CSV files for direct use in the oil-market analysis project.

### Series definition used

- Source: **EIA International**
- Frequency: **monthly**
- Product: **Crude oil including lease condensate**
- EIA product ID: `57`
- Activity: **Production**
- EIA activity ID: `1`
- Unit: **TBPD = thousand barrels per day**

> **Important:** “Non-OPEC” is not the same as “OPEC+”. Russia, Kazakhstan, Oman, Azerbaijan, etc. remain included because they are not OPEC members.

> The OPEC membership filter below uses the current 12-member composition. If you are doing a long historical study where membership should change through time, use a date-aware membership table instead.


## 1. Setup

You need a free EIA API key.

Packages used: `requests`, `pandas`, `numpy`, and `pycountry`.

If a package is missing, install it once in your environment, for example:

```bash
pip install requests pandas numpy pycountry
```


In [ ]:
import requests
import pandas as pd
import numpy as np
import pycountry
from pathlib import Path

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)


## 2. Parameters

Change the API key and date range here.

For your existing monthly oil-market project, set `START_DATE` and `END_DATE` to the same period used by your other features before merging.


In [ ]:
EIA_API_KEY = "kMWwLaP6bX0Qwf2zdKR4cE4hVt4r2cDQjpRZJeCg"

START_DATE = "2000-01"
END_DATE = None          # Example: "2026-04". Keep None for latest available month.

PRODUCT_ID = "57"       # Crude oil including lease condensate
ACTIVITY_ID = "1"       # Production
UNIT = "TBPD"           # Thousand barrels per day
FREQUENCY = "monthly"

BASE_URL = "https://api.eia.gov/v2/international"
PAGE_SIZE = 5000

OUTPUT_DIR = Path("./non_opec_crude_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 3. Define Current OPEC Members

As of 2026, OPEC has 12 members. These country codes are excluded from the country-level data.

Angola is **not** excluded because it left OPEC at the end of 2023.


In [ ]:
OPEC_MEMBER_CODES = {
    "DZA",  # Algeria
    "COG",  # Republic of the Congo
    "GNQ",  # Equatorial Guinea
    "GAB",  # Gabon
    "IRN",  # Iran
    "IRQ",  # Iraq
    "KWT",  # Kuwait
    "LBY",  # Libya
    "NGA",  # Nigeria
    "SAU",  # Saudi Arabia
    "ARE",  # United Arab Emirates
    "VEN",  # Venezuela
}

OPEC_MEMBER_CODES


## 4. EIA API Helpers

The EIA API returns at most 5,000 JSON rows per response, so the fetch function below automatically paginates until all matching rows are downloaded.


In [ ]:
def _check_api_key(api_key: str) -> None:
    if not api_key or api_key == "YOUR_EIA_API_KEY":
        raise ValueError(
            "Add your EIA API key to EIA_API_KEY before running the download cells."
        )


def fetch_eia_facet(facet_id: str, api_key: str) -> pd.DataFrame:
    """Download the available values for one EIA International facet."""
    _check_api_key(api_key)

    url = f"{BASE_URL}/facet/{facet_id}/"
    params = {
        "api_key": api_key,
        "length": 5000,
    }

    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()
    payload = response.json()

    if "error" in payload:
        raise RuntimeError(payload["error"])

    return pd.DataFrame(payload["response"]["facets"])


def fetch_eia_international_monthly(
    api_key: str,
    start: str | None = None,
    end: str | None = None,
    country_region_type_id: str | None = None,
) -> pd.DataFrame:
    """Fetch monthly EIA International crude production with automatic pagination."""
    _check_api_key(api_key)

    url = f"{BASE_URL}/data/"
    params = {
        "api_key": api_key,
        "frequency": FREQUENCY,
        "data[0]": "value",
        "facets[activityId][]": ACTIVITY_ID,
        "facets[productId][]": PRODUCT_ID,
        "facets[unit][]": UNIT,
        "sort[0][column]": "period",
        "sort[0][direction]": "asc",
        "length": PAGE_SIZE,
    }

    if start:
        params["start"] = start
    if end:
        params["end"] = end
    if country_region_type_id:
        params["facets[countryRegionTypeId][]"] = country_region_type_id

    all_rows = []
    offset = 0

    while True:
        params["offset"] = offset

        response = requests.get(url, params=params, timeout=90)
        response.raise_for_status()
        payload = response.json()

        if "error" in payload:
            raise RuntimeError(payload["error"])

        response_block = payload["response"]
        rows = response_block.get("data", [])
        total = int(response_block.get("total", len(rows)))

        if not rows:
            break

        all_rows.extend(rows)
        offset += len(rows)

        print(f"Fetched {offset:,} / {total:,} rows", end="\r")

        if offset >= total:
            break

    print(f"Fetched {len(all_rows):,} rows total.")
    return pd.DataFrame(all_rows)


## 5. Identify the EIA “Country” Geography Type

The International API can also contain regional/aggregate geographies. We first ask EIA which `countryRegionTypeId` corresponds to countries and use it directly in the data request.

If that metadata is unavailable, the cleaning section has an ISO-3 fallback.


In [ ]:
country_types = fetch_eia_facet("countryRegionTypeId", EIA_API_KEY)
display(country_types)

country_type_id = None

if not country_types.empty:
    searchable_cols = [
        c for c in ["name", "alias", "description"]
        if c in country_types.columns
    ]

    if searchable_cols:
        text = (
            country_types[searchable_cols]
            .fillna("")
            .astype(str)
            .agg(" ".join, axis=1)
            .str.lower()
        )

        matches = country_types[text.str.contains(r"\bcountry\b", regex=True)]

        if not matches.empty:
            country_type_id = str(matches.iloc[0]["id"])

print("Detected countryRegionTypeId:", country_type_id)


## 6. Download Monthly Crude Oil Production

This downloads the EIA country-level production observations for the chosen period.


In [ ]:
raw = fetch_eia_international_monthly(
    api_key=EIA_API_KEY,
    start=START_DATE,
    end=END_DATE,
    country_region_type_id=country_type_id,
)

print("Shape:", raw.shape)
display(raw.head())
display(raw.tail())


## 7. Clean the Country-Level Data

The EIA response normally includes country code/name fields. We standardize them and keep only valid country-level ISO-3 records.

The ISO-3 fallback prevents aggregate regions such as “World” from entering the country sum if the geography-type filter was unavailable.


In [ ]:
df = raw.copy()

# Standardize date and value
df["date"] = pd.to_datetime(df["period"], format="%Y-%m", errors="coerce")
df["value"] = pd.to_numeric(df["value"], errors="coerce")

# Identify EIA country columns
country_code_col = "countryRegionId"

possible_name_cols = [
    "countryRegionName",
    "countryRegionDescription",
    "countryRegion",
]
country_name_col = next(
    (c for c in possible_name_cols if c in df.columns),
    None
)

if country_code_col not in df.columns:
    raise KeyError(
        f"Expected '{country_code_col}' in the EIA response. "
        f"Available columns: {list(df.columns)}"
    )

# If EIA did not successfully restrict the request to country records,
# keep only current ISO-3166 alpha-3 country codes as a fallback.
valid_iso3 = {country.alpha_3 for country in pycountry.countries}
df = df[df[country_code_col].isin(valid_iso3)].copy()

if country_name_col is None:
    def iso_name(code):
        country = pycountry.countries.get(alpha_3=code)
        return country.name if country else code

    df["country"] = df[country_code_col].map(iso_name)
else:
    df["country"] = df[country_name_col].astype(str)

df = df.rename(
    columns={
        country_code_col: "country_code",
        "value": "production_tbpd",
    }
)

df = (
    df[["date", "country_code", "country", "production_tbpd"]]
    .dropna(subset=["date", "production_tbpd"])
    .drop_duplicates(subset=["date", "country_code"], keep="last")
    .sort_values(["date", "country"])
    .reset_index(drop=True)
)

print("Country-level rows:", len(df))
print("Countries found:", df["country_code"].nunique())
print("Date range:", df["date"].min(), "to", df["date"].max())
display(df.head(10))


## 8. Remove OPEC Countries

This produces the actual **non-OPEC country dataset**.

Russia, Canada, Brazil, China, Norway, Kazakhstan, Oman, Mexico, the United States, etc. remain included.


In [ ]:
non_opec = df[
    ~df["country_code"].isin(OPEC_MEMBER_CODES)
].copy()

non_opec = non_opec.sort_values(
    ["date", "country"]
).reset_index(drop=True)

print("Non-OPEC rows:", len(non_opec))
print("Non-OPEC countries:", non_opec["country_code"].nunique())
print(
    "OPEC codes still present:",
    sorted(set(non_opec["country_code"]) & OPEC_MEMBER_CODES)
)

display(non_opec.head(10))


## 9. Quick Validation

For the latest available month, inspect the largest non-OPEC crude producers. This is a simple sanity check that major producers such as the United States, Russia, Canada, China, and Brazil appear where data are available.


In [ ]:
latest_date = non_opec["date"].max()

latest_month = (
    non_opec[non_opec["date"] == latest_date]
    .sort_values("production_tbpd", ascending=False)
)

print("Latest available month:", latest_date.strftime("%Y-%m"))
display(latest_month.head(20))


## 10. Create the Monthly Non-OPEC Total

This aggregates all retained non-OPEC country series into one monthly feature.

`country_count_reporting` is kept as a coverage diagnostic. A sudden fall in the number of reporting countries can make a total look artificially low, so always inspect this column before interpreting unusual changes.


In [ ]:
non_opec_total = (
    non_opec.groupby("date", as_index=False)
    .agg(
        non_opec_total_production_tbpd=("production_tbpd", "sum"),
        country_count_reporting=("country_code", "nunique"),
    )
    .sort_values("date")
    .reset_index(drop=True)
)

display(non_opec_total.head())
display(non_opec_total.tail())


## 11. Create Wide Country Features

This shape is convenient if you want individual country production values as model features.

Example columns:

- `production_USA_tbpd`
- `production_RUS_tbpd`
- `production_CAN_tbpd`
- `production_BRA_tbpd`
- ...


In [ ]:
non_opec_wide = (
    non_opec.pivot(
        index="date",
        columns="country_code",
        values="production_tbpd",
    )
    .sort_index()
)

non_opec_wide.columns = [
    f"production_{code}_tbpd"
    for code in non_opec_wide.columns
]

non_opec_wide = non_opec_wide.reset_index()

display(non_opec_wide.head())
print("Wide shape:", non_opec_wide.shape)


## 12. Merge the Total Back into the Wide Dataset

For your oil-market project, this gives you both individual non-OPEC country production features and the overall non-OPEC production feature in the same monthly table.


In [ ]:
non_opec_features = non_opec_wide.merge(
    non_opec_total,
    on="date",
    how="left",
)

display(non_opec_features.head())
print("Final feature-table shape:", non_opec_features.shape)


## 13. Export CSV Files


In [ ]:
long_path = OUTPUT_DIR / "non_opec_crude_production_monthly_long.csv"
wide_path = OUTPUT_DIR / "non_opec_crude_production_monthly_wide.csv"
total_path = OUTPUT_DIR / "non_opec_total_crude_production_monthly.csv"
features_path = OUTPUT_DIR / "non_opec_crude_production_monthly_features.csv"

non_opec.to_csv(long_path, index=False)
non_opec_wide.to_csv(wide_path, index=False)
non_opec_total.to_csv(total_path, index=False)
non_opec_features.to_csv(features_path, index=False)

print("Saved:")
print(" -", long_path)
print(" -", wide_path)
print(" -", total_path)
print(" -", features_path)


## 14. Optional: Merge Only the Non-OPEC Total into Your Existing Dataset

If your existing oil-market dataframe is called `final_df` and already contains one monthly row per date, this is probably the cleanest feature to add.

Your `final_df["date"]` must represent the same monthly period.


In [ ]:
kMWwLaP6bX0Qwf2zdKR4cE4hVt4r2cDQjpRZJeCg

## 15. Data-Quality Checks Before Analysis

Run these checks before using the feature in correlations, regression, forecasting, or causal investigation.


In [ ]:
quality_summary = pd.Series({
    "start_month": non_opec["date"].min(),
    "end_month": non_opec["date"].max(),
    "country_count": non_opec["country_code"].nunique(),
    "rows": len(non_opec),
    "missing_production_values": non_opec["production_tbpd"].isna().sum(),
    "duplicate_month_country_rows": non_opec.duplicated(
        subset=["date", "country_code"]
    ).sum(),
    "min_reporting_countries_per_month": non_opec_total[
        "country_count_reporting"
    ].min(),
    "max_reporting_countries_per_month": non_opec_total[
        "country_count_reporting"
    ].max(),
})

display(quality_summary.to_frame("value"))


## Notes for the Oil-Market Project

- Use the **same definition and units** when comparing OPEC and non-OPEC production.
- The EIA series here is **crude oil including lease condensate**, measured in **thousand barrels per day**.
- Do not compare this directly with a broader “petroleum and other liquids” consumption series without understanding the product-definition difference.
- For a compact market model, `non_opec_total_production_tbpd` is usually more useful than hundreds of individual country columns.
- For supply-driver analysis, retain major country features such as the United States, Russia, Canada, Brazil, China, Norway, Kazakhstan, Mexico, and Oman.
- If your analysis spans periods when countries joined or left OPEC, replace the static current-member filter with date-aware membership logic.
